# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [2]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   object 
 1   continent  1704 non-null   object 
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   object 
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 106.6+ KB
None
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: ['Asia' 'Europe' 'Africa' 'Americas' 'Oceania']
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.960121e+07     7215.3    

## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2002 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [3]:
# Task 1
import plotly.graph_objects as go

# Chosen continent: Europe
YEARS = [2002, 2007]
eu = df.loc[(df['continent'] == 'Europe') & (df['year'].isin(YEARS))].copy()
eu['year'] = eu['year'].astype(str)          # categorical -> exactly 2 colours

fig1 = px.scatter(
    eu, x='gdpPercap', y='lifeExp', color='year',
    hover_name='country', log_x=True,
    color_discrete_map={'2002': '#9ecae1', '2007': '#08519c'},   # light -> dark = earlier -> later
    category_orders={'year': ['2002', '2007']},
    labels={'gdpPercap': 'GDP per capita (US$, log scale)',
            'lifeExp': 'Life expectancy (years)', 'year': 'Year'},
)

# Change direction: a thin grey arrow from each country's 2002 point to its 2007 point
wide = eu.pivot(index='country', columns='year', values=['gdpPercap', 'lifeExp'])
for country, r in wide.iterrows():
    fig1.add_annotation(
        x=np.log10(r[('gdpPercap', '2007')]), y=r[('lifeExp', '2007')],
        ax=np.log10(r[('gdpPercap', '2002')]), ay=r[('lifeExp', '2002')],
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1, arrowcolor='rgba(120,120,120,0.6)',
    )

fig1.update_traces(marker=dict(size=9, line=dict(width=0.5, color='white')))
fig1.update_layout(
    title=dict(text='Every European country got richer and longer-lived between 2002 and 2007<br>'
                    '<sup>Arrows run from 2002 to 2007; all 30 countries moved up and to the right</sup>'),
    template='plotly_white', width=900, height=550,
    legend=dict(title='', orientation='h', y=1.08, x=1, xanchor='right'),
)
fig1.show()

## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [4]:
# Task 2
HIGHLIGHT = 'Asia'
d07 = df.loc[df['year'] == 2007].copy()
d07['group'] = np.where(d07['continent'] == HIGHLIGHT, HIGHLIGHT, 'Other continents')

fig2 = px.scatter(
    d07.sort_values('group', ascending=False),     # draw 'Other' first so Asia sits on top
    x='gdpPercap', y='lifeExp', size='pop', color='group',
    hover_name='country', log_x=True, size_max=60,
    color_discrete_map={HIGHLIGHT: '#d95f02', 'Other continents': '#cccccc'},
    category_orders={'group': ['Other continents', HIGHLIGHT]},
    labels={'gdpPercap': 'GDP per capita (US$, log scale)',
            'lifeExp': 'Life expectancy (years)', 'group': ''},
)
fig2.update_traces(marker=dict(line=dict(width=0.5, color='white'), opacity=0.8))

# Annotations: the highlighted group's story
asia = d07[d07['continent'] == HIGHLIGHT]
def pt(name):
    r = asia.loc[asia['country'] == name].iloc[0]
    return np.log10(r['gdpPercap']), r['lifeExp']

for name, text, ax_, ay_ in [
    ('China', 'China & India: ~2.4 billion people<br>in the mid-income range', -90, 60),
    ('Japan', 'Japan: highest life expectancy<br>in the world (82.6 yrs)', 0, -50),
]:
    x, y = pt(name)
    fig2.add_annotation(x=x, y=y, text=text, showarrow=True, arrowhead=2,
                        ax=ax_, ay=ay_, font=dict(color='#d95f02', size=12),
                        arrowcolor='#d95f02', bgcolor='white')

fig2.update_layout(
    title=dict(text='Asia spans the whole range of wealth and health, and holds most of the world\'s people<br>'
                    '<sup>2007; bubble size = population</sup>'),
    template='plotly_white', width=950, height=600,
    legend=dict(orientation='h', y=1.08, x=1, xanchor='right'),
)
fig2.update_traces(selector=dict(name='Other continents'), showlegend=True)
fig2.show()